In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/external-catalog-1/default/external-volume-1/employee.csv")
display(df)

In [0]:
from pyspark.sql.functions import *
df1= df.filter(col("dept_id") == 100)
df1.write \
    .format ("delta") \
    .mode("overwrite") \
    .saveAsTable ("`external-catalog-1`.default.employees_delta")
print("Write successfull")

In [0]:
df_delta = spark.read.table("`external-catalog-1`.default.employees_delta")
df_delta.show(2)

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "`external-catalog-1`.default.employees_delta")
history_df = deltaTable.history()
display(history_df.select("version"))

In [0]:
history_sql = """
DESCRIBE HISTORY `external-catalog-1`.default.employees_delta
"""
history_df_sql = spark.sql(history_sql)
display(history_df_sql.select("version","timestamp","operation"))

In [0]:
deltaTable.delete(col("salary") == 6000)
df_delta = spark.read.table("`external-catalog-1`.default.employees_delta")
display(df_delta)

In [0]:
dummy_record = [("999", "John Doe", "100", "6000")]
columns = ["emp_id", "emp_name", "dept_id", "salary"]
dummy_df = spark.createDataFrame(dummy_record, columns)

dummy_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("`external-catalog-1`.default.employees_delta")

df_delta = spark.read.table("`external-catalog-1`.default.employees_delta")
display(df_delta)

In [0]:
history_sql = """
DESCRIBE HISTORY `external-catalog-1`.default.employees_delta
"""
history_df_sql = spark.sql(history_sql)
display(history_df_sql.select("version","timestamp","operation"))

In [0]:
df_delta_v1 = spark.read.option("versionAsOf", 1).table("`external-catalog-1`.default.employees_delta")
display(df_delta_v1)